# HAND Beginner EMG Analysis

This lab works with a **synthetic MyoWare-style analog envelope**, not raw EMG. The sensor module has already amplified, rectified, and enveloped the small biological signal. Our software must turn that changing voltage into a conservative `REST` or `FLEX` intent.

By the end, you will have plotted the signal, smoothed noise, estimated a baseline, normalized the signal, selected thresholds, applied hysteresis and debounce, and measured classification errors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

candidates = [
    Path("../data/synthetic_emg_envelope.csv"),
    Path("onboarding/emg/data/synthetic_emg_envelope.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find synthetic_emg_envelope.csv")

data_path

## 1. Load and inspect

Always verify columns, missing values, time ordering, and sampling rate before filtering a signal.

In [ ]:
df = pd.read_csv(data_path)
required_columns = {"time_s", "envelope_v", "label", "event"}
assert required_columns.issubset(df.columns)
assert df[list(required_columns)].notna().all().all()
assert df["time_s"].is_monotonic_increasing

sample_period_s = df["time_s"].diff().dropna().median()
sample_rate_hz = 1.0 / sample_period_s
print(f"Rows: {len(df):,}")
print(f"Estimated sample rate: {sample_rate_hz:.1f} Hz")
print(df["label"].value_counts())
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df["time_s"], df["envelope_v"], linewidth=1, label="Envelope voltage")
ax.fill_between(
    df["time_s"], 0, 3.3, where=df["label"].eq("flex"),
    color="tab:green", alpha=0.12, label="Ground-truth flex"
)
ax.set(xlabel="Time (s)", ylabel="Envelope (V)", ylim=(0, 2.1), title="Synthetic EMG envelope")
ax.legend(loc="upper right")
plt.show()

## 2. Smooth, estimate the resting baseline, and normalize

A 200 ms rolling mean suppresses short noise. We estimate baseline only from the first five seconds, which are our simulated rest calibration window. Subtracting that baseline makes the activation value easier to interpret. Look into later applying a Kalman filter (not in the scope of this onboarding example).

In [ ]:
smoothing_ms = 200
window_samples = max(1, round(sample_rate_hz * smoothing_ms / 1000))
df["smoothed_v"] = df["envelope_v"].rolling(
    window=window_samples, center=True, min_periods=1
).mean()

rest_calibration = df["time_s"] < 5.0
baseline_v = df.loc[rest_calibration, "smoothed_v"].median()
df["activation_v"] = (df["smoothed_v"] - baseline_v).clip(lower=0)

print(f"Smoothing window: {window_samples} samples ({smoothing_ms} ms)")
print(f"Estimated resting baseline: {baseline_v:.3f} V")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df["time_s"], df["envelope_v"], alpha=0.35, linewidth=0.8, label="Raw envelope")
ax.plot(df["time_s"], df["smoothed_v"], linewidth=2, label="200 ms rolling mean")
ax.axhline(baseline_v, color="black", linestyle="--", label="Rest baseline")
ax.set(xlabel="Time (s)", ylabel="Voltage (V)", title="Smoothing and baseline")
ax.legend()
plt.show()

## 3. Calibrate thresholds

We use the first rest window and the middle of the first flex as simple calibration trials. The `ON` threshold lies between the noisy high end of rest and the weak low end of flex. The lower `OFF` threshold creates **hysteresis**, preventing rapid switching near one threshold.

In [ ]:
flex_calibration = df["time_s"].between(6.5, 8.5)
rest_high_v = df.loc[rest_calibration, "activation_v"].quantile(0.99)
flex_low_v = df.loc[flex_calibration, "activation_v"].quantile(0.10)
on_threshold_v = (rest_high_v + flex_low_v) / 2
off_threshold_v = 0.70 * on_threshold_v

print(f"99th-percentile rest activation: {rest_high_v:.3f} V")
print(f"10th-percentile flex activation: {flex_low_v:.3f} V")
print(f"ON threshold: {on_threshold_v:.3f} V")
print(f"OFF threshold: {off_threshold_v:.3f} V")

## 4. Apply hysteresis and debounce

Hysteresis uses separate ON and OFF thresholds. Debounce additionally requires the condition to remain true for several samples. Both help reject chatter and short artifacts.

In [ ]:
on_debounce_ms = 140
off_debounce_ms = 180
on_samples = max(1, round(sample_rate_hz * on_debounce_ms / 1000))
off_samples = max(1, round(sample_rate_hz * off_debounce_ms / 1000))

active = False
above_count = 0
below_count = 0
predicted = []
for value in df["activation_v"]:
    if not active:
        above_count = above_count + 1 if value >= on_threshold_v else 0
        if above_count >= on_samples:
            active = True
            below_count = 0
    else:
        below_count = below_count + 1 if value <= off_threshold_v else 0
        if below_count >= off_samples:
            active = False
            above_count = 0
    predicted.append(active)

df["predicted_flex"] = predicted
truth = df["label"].eq("flex")
prediction = df["predicted_flex"]
tp = int((truth & prediction).sum())
tn = int((~truth & ~prediction).sum())
fp = int((~truth & prediction).sum())
fn = int((truth & ~prediction).sum())
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0

print({"true_positive": tp, "true_negative": tn, "false_positive": fp, "false_negative": fn})
print(f"Precision: {precision:.3f}; recall: {recall:.3f}")
print(f"False-active time: {fp / sample_rate_hz:.2f} s")

In [ ]:
fig, (signal_ax, state_ax) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
signal_ax.plot(df["time_s"], df["activation_v"], label="Normalized activation")
signal_ax.axhline(on_threshold_v, color="tab:red", linestyle="--", label="ON threshold")
signal_ax.axhline(off_threshold_v, color="tab:orange", linestyle=":", label="OFF threshold")
signal_ax.set(ylabel="Activation (V)", title="Thresholded intent detector")
signal_ax.legend(loc="upper right")

state_ax.fill_between(df["time_s"], 0, 1, where=truth, step="post", alpha=0.25, label="True flex")
state_ax.step(df["time_s"], prediction.astype(int), where="post", color="tab:red", label="Predicted flex")
state_ax.set(xlabel="Time (s)", ylabel="State", yticks=[0, 1], yticklabels=["REST", "FLEX"])
state_ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. Reflection and experiments

Answer these in a new Markdown cell:

1. Where does the detector respond late, and which smoothing/debounce choices cause that delay?
2. Do the two motion artifacts cause false activations? Why or why not?
3. What happens when smoothing is changed from 200 ms to 50 ms or 500 ms?
4. What is the tradeoff when the ON threshold is increased by 25%?
5. Why should the first physical version produce a discrete command instead of mapping EMG voltage directly to gripping force?
6. Which assumptions in this synthetic exercise must be retested for every real wearer/session?

The goal is not perfect synthetic accuracy. The goal is to explain the delay-versus-false-activation tradeoff and identify what must be recalibrated with real data.